In [1]:
from data_loading.models_dataset import ArchiMateDataset, EcoreDataset


reload_db = True

config_params = dict(
    min_enr = 1.2,
    min_edges = 10,
)
ecore = EcoreDataset('ecore_555', reload=reload_db, **config_params)
modelset = EcoreDataset('modelset', reload=reload_db, remove_duplicates=True, **config_params)
mar = EcoreDataset('mar-ecore-github', reload=reload_db, **config_params)
eamodelset = ArchiMateDataset('eamodelset', reload=reload_db, **config_params)

/home/sali/CMAI-Projects/cm2ml-encodings-eval/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset exists: True, reload: True


Loading Ecore_555: 100%|██████████| 548/548 [00:01<00:00, 452.70it/s]


Loaded Total ecore_555 with 548 graphs
Filtering...
Saving ecore_555 to pickle
Saved ecore_555 to pickle
Loaded ecore_555 with 354 graphs
Dataset exists: True, reload: True


Loading Modelset: 100%|██████████| 4127/4127 [00:03<00:00, 1361.63it/s]


Loaded Total modelset with 2043 graphs
Filtering...
Saving modelset to pickle
Saved modelset to pickle
Loaded modelset with 845 graphs
Dataset exists: True, reload: True


Loading Mar-Ecore-Github: 100%|██████████| 18110/18110 [00:31<00:00, 574.85it/s] 


Loaded Total mar-ecore-github with 18110 graphs
Filtering...
Saving mar-ecore-github to pickle
Saved mar-ecore-github to pickle
Loaded mar-ecore-github with 7036 graphs


Loading Eamodelset: 100%|██████████| 979/979 [00:03<00:00, 272.07it/s]


Total graphs: 969
Saving eamodelset to pickle
Saved eamodelset to pickle
Loaded eamodelset with 456 graphs
Graphs: 456


In [6]:
g = min([g for g in ecore.graphs if g is not None], key=lambda g: g.number_of_edges())

In [10]:
g.nodes(data=True)

NodeDataView({'Scheme': {'name': 'Scheme', 'attributes': ['name'], 'abstract': False}, 'Table': {'name': 'Table', 'attributes': ['name'], 'abstract': False}, 'Column': {'name': 'Column', 'attributes': ['name'], 'abstract': False}, 'FKey': {'name': 'FKey', 'attributes': [], 'abstract': False}, 'PKey': {'name': 'PKey', 'attributes': [], 'abstract': False}})

In [11]:
g.edges(data=True)

OutEdgeDataView([('Scheme', 'Table', {'name': 'tables', 'type': 'containment'}), ('Scheme', 'FKey', {'name': 'keys', 'type': 'containment'}), ('Table', 'Column', {'name': 'columns', 'type': 'containment'}), ('Table', 'Scheme', {'name': 'scheme', 'type': 'reference'}), ('Table', 'PKey', {'name': 'key', 'type': 'containment'}), ('Column', 'Table', {'name': 'table', 'type': 'reference'}), ('FKey', 'PKey', {'name': 'refersTo', 'type': 'reference'}), ('FKey', 'Column', {'name': 'column', 'type': 'reference'}), ('FKey', 'Scheme', {'name': 'scheme', 'type': 'reference'}), ('PKey', 'Column', {'name': 'column', 'type': 'reference'})])

In [ ]:
import pandas as pd

df = pd.read_csv("results/analysis_rigorous/evaluation_package/eamodelset_layer/flat_results.csv")
df.drop(columns=['dataset', 'encoder_name', 'target_attr', 'label_attr', 'attributes_attr', 'use_attributes', 'use_edge_label', 'graph_encoder_family', 'structural_encoding_label', 'duplicate_count', 'num_samples', 'num_classes'], inplace=True)

In [4]:
df.columns

Index(['encoder_key', 'classification_mode', 'tree', 'path_depth',
       'use_edge_type', 'gnn_model_name', 'accuracy', 'precision_macro',
       'recall_macro', 'f1_macro'],
      dtype='str')

In [5]:
import pandas as pd

encoders = ['bow', 'tfidf', 'w2v', 'bert-encoder']
cls_mode = ['gnn', 'text']
booleans = ["tree", "use_edge_type"]
gnn_models = ['gcn', 'gat', 'sage']

metric_columns = ['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
parameter_columns = ['encoder_key', 'classification_mode', 'tree', 'use_edge_type', 'gnn_model_name']


def get_unique_values(df, parameter):
    if parameter == "encoder_key":
        return encoders
    elif parameter == "classification_mode":
        return cls_mode
    elif parameter in ["tree", "use_edge_type"]:
        return [False, True]
    elif parameter == "gnn_model_name":
        return gnn_models
    else:
        raise ValueError(f"Unknown parameter: {parameter}")


def make_config_key(df, fixed_columns, sep="||"):
    """
    Create a stable config key from fixed columns.
    """
    return df[fixed_columns].astype(str).agg(sep.join, axis=1)


def get_transition_groups(df, parameter_of_interest):
    """
    Creates transition groups by:
    1. Identifying fixed columns = all parameter columns except parameter_of_interest
    2. Creating a dummy config_key from fixed columns
    3. Grouping by config_key
    4. Within each group, comparing consecutive values of parameter_of_interest
       based on the predefined order from get_unique_values()

    Output:
    [
        {
            "transition": ("bow", "tfidf"),
            "config_key": "gnn||False||True||gcn",
            "fixed_params": {
                "classification_mode": "gnn",
                "tree": False,
                "use_edge_type": True,
                "gnn_model_name": "gcn"
            },
            "delta_report": {
                "before": {...},
                "after": {...},
                "delta": {...}
            }
        },
        ...
    ]
    """
    df = df.copy()

    unique_values = get_unique_values(df, parameter_of_interest)
    transitions = list(zip(unique_values[:-1], unique_values[1:]))

    fixed_columns = [col for col in parameter_columns if col != parameter_of_interest]
    if parameter_of_interest == "classification_mode":
        fixed_columns.remove("gnn_model_name")  # gnn_model_name is irrelevant when classification_mode is not gnn
        
    df["config_key"] = make_config_key(df, fixed_columns)

    results = []

    for config_key, group in df.groupby("config_key", dropna=False):
        # recover fixed params from first row
        first_row = group.iloc[0]
        fixed_params = {col: first_row[col] for col in fixed_columns}

        # map each parameter value to its row
        # if duplicates exist, this keeps the last one; aggregate first if needed
        value_to_row = {
            row[parameter_of_interest]: row
            for _, row in group.iterrows()
        }

        for before_val, after_val in transitions:
            if before_val in value_to_row and after_val in value_to_row:
                before_row = value_to_row[before_val]
                after_row = value_to_row[after_val]

                before_metrics = {
                    metric: float(before_row[metric]) for metric in metric_columns
                }
                after_metrics = {
                    metric: float(after_row[metric]) for metric in metric_columns
                }
                delta_metrics = {
                    metric: after_metrics[metric] - before_metrics[metric]
                    for metric in metric_columns
                }

                results.append({
                    "transition": (before_val, after_val),
                    "config_key": config_key,
                    "fixed_params": fixed_params,
                    "delta_report": {
                        "before": before_metrics,
                        "after": after_metrics,
                        "delta": delta_metrics
                    }
                })

    return results

In [6]:
reports = get_transition_groups(df, "classification_mode")
print(f"Total transitions found: {len(reports)}")
for r in reports[:3]:
    print(r)

Total transitions found: 16
{'transition': ('gnn', 'text'), 'config_key': 'bert-encoder||False||False', 'fixed_params': {'encoder_key': 'bert-encoder', 'tree': np.False_, 'use_edge_type': np.False_}, 'delta_report': {'before': {'accuracy': 0.6868020304568528, 'precision_macro': 0.7544986996548336, 'recall_macro': 0.4790027326525348, 'f1_macro': 0.5264235112842338}, 'after': {'accuracy': 0.6482233502538071, 'precision_macro': 0.6095774869238811, 'recall_macro': 0.530700874374624, 'f1_macro': 0.5630727066917179}, 'delta': {'accuracy': -0.03857868020304578, 'precision_macro': -0.1449212127309525, 'recall_macro': 0.0516981417220892, 'f1_macro': 0.036649195407484125}}}
{'transition': ('gnn', 'text'), 'config_key': 'bert-encoder||False||True', 'fixed_params': {'encoder_key': 'bert-encoder', 'tree': np.False_, 'use_edge_type': np.True_}, 'delta_report': {'before': {'accuracy': 0.6872250423011844, 'precision_macro': 0.7597208252227364, 'recall_macro': 0.4792055523779387, 'f1_macro': 0.52579944

In [7]:
def filter_transition_groups(reports, filter_params):
    """
    Filter transition groups based on fixed parameters.

    filter_params: dict of {param_name: desired_value}
    Example: {"tree": True, "use_edge_type": False}

    Returns a list of transitions that match the filter criteria.
    """
    filtered = []
    for report in reports:
        fixed_params = report["fixed_params"]
        if all(fixed_params.get(param) == value for param, value in filter_params.items()):
            filtered.append(report)
    return filtered

In [8]:
print("\nTransitions with tree encoding:")
with_tree_reports = filter_transition_groups(reports, {
    # "classification_mode": "gnn",
    "tree": True,
    # "use_edge_type": True,
    # "gnn_model_name": "gcn"
})

for r in with_tree_reports:
    print(r['transition'], end=": ")
    print(r['config_key'])
    # print(r['delta_report']['before']['f1_macro'])
    # print(r['delta_report']['after']['f1_macro'])
    print(r['delta_report']['delta'])

without_tree_reports = filter_transition_groups(reports, {
    # "classification_mode": "gnn",
    "tree": False,
    # "use_edge_type": True,
    # "gnn_model_name": "gcn"
})
print("\nTransitions without tree encoding:")
for r in without_tree_reports:
    print(r['transition'], end=": ")
    print(r['config_key'])
    # print(r['delta_report']['before']['f1_macro'])
    # print(r['delta_report']['after']['f1_macro'])
    print(r['delta_report']['delta'])



Transitions with tree encoding:
('gnn', 'text'): bert-encoder||True||False
{'accuracy': 0.2036379018612522, 'precision_macro': 0.5374466726396439, 'recall_macro': 0.3002950692936258, 'f1_macro': 0.4070640654801462}
('gnn', 'text'): bert-encoder||True||True
{'accuracy': 0.2036379018612522, 'precision_macro': 0.5374466726396439, 'recall_macro': 0.3002950692936258, 'f1_macro': 0.4070640654801462}
('gnn', 'text'): bow||True||False
{'accuracy': 0.4314720812182742, 'precision_macro': 0.7231407739365828, 'recall_macro': 0.40164429328287066, 'f1_macro': 0.5500135010287829}
('gnn', 'text'): bow||True||True
{'accuracy': 0.4314720812182742, 'precision_macro': 0.7231407739365828, 'recall_macro': 0.40164429328287066, 'f1_macro': 0.5500135010287829}
('gnn', 'text'): tfidf||True||False
{'accuracy': 0.422419627749577, 'precision_macro': 0.7266153668357535, 'recall_macro': 0.37845345694617094, 'f1_macro': 0.5263667886663407}
('gnn', 'text'): tfidf||True||True
{'accuracy': 0.422419627749577, 'precision

In [9]:
import pandas as pd

metric_columns = ["accuracy", "precision_macro", "recall_macro", "f1_macro"]

# Ordered values define the transition direction.
# Change the order if you want the delta sign to flip.
FACTOR_SPECS = {
    "encoder_key": {
        "values": ["bow", "tfidf", "w2v", "bert-encoder"],
        "requires": {}
    },
    "classification_mode": {
        "values": ["gnn", "text"],   # current direction matches your example output
        "requires": {}
    },
    "use_edge_type": {
        "values": [False, True],
        "requires": {}
    },
    "path_depth": {
        "values": [0, 1, 2, 3],
        "requires": {}
    },
    "gnn_model_name": {
        "values": ["gcn", "gat", "sage"],
        "requires": {"classification_mode": "gnn"}
    },
    # Optional if these columns exist in your dataframe
    "use_attributes": {
        "values": [False, True],
        "requires": {}
    },
    "use_edge_label": {
        "values": [False, True],
        "requires": {}
    },
    # tree is handled separately below
    "tree": {
        "values": [False, True],
        "requires": {}
    },
}


def make_config_key(df, fixed_columns, sep="||"):
    if not fixed_columns:
        return pd.Series(["__all__"] * len(df), index=df.index)
    return df[fixed_columns].astype(str).agg(sep.join, axis=1)


def deduplicate_configs(df, parameter_columns):
    # Keeps the last row for each exact configuration
    return df.drop_duplicates(subset=parameter_columns, keep="last").copy()


def get_transitions_for_factor(
    df,
    factor,
    parameter_columns,
    metric_columns,
    tree_value=None,
):
    """
    Matched transition analysis for one factor.
    If tree_value is not None, analysis is restricted to that tree stratum.
    """
    if factor not in FACTOR_SPECS:
        raise ValueError(f"Unknown factor: {factor}")

    if factor not in df.columns:
        return []

    work = df.copy()

    # Keep tree-stratified analyses separate
    if tree_value is not None:
        if "tree" not in work.columns:
            raise ValueError("Column 'tree' not found in dataframe.")
        work = work[work["tree"] == tree_value].copy()

    # Factor-specific applicability filters
    for col, required_val in FACTOR_SPECS[factor]["requires"].items():
        if col in work.columns:
            work = work[work[col] == required_val].copy()

    if work.empty:
        print(f"No data left for factor '{factor}' with tree={tree_value} after applying filters.")
        return []

    ordered_values = [v for v in FACTOR_SPECS[factor]["values"] if v in set(work[factor].dropna().tolist())]
    transitions = list(zip(ordered_values[:-1], ordered_values[1:]))

    # Fixed columns = all parameters except factor-of-interest
    fixed_columns = [col for col in parameter_columns if col != factor and col in work.columns]

    # Tree is fixed externally when tree_value is set
    if tree_value is not None and "tree" in fixed_columns:
        fixed_columns.remove("tree")

    # gnn_model_name is irrelevant when comparing classification_mode
    if factor == "classification_mode" and "gnn_model_name" in fixed_columns:
        fixed_columns.remove("gnn_model_name")

    work["config_key"] = make_config_key(work, fixed_columns)

    results = []

    for config_key, group in work.groupby("config_key", dropna=False):
        first_row = group.iloc[0]
        fixed_params = {col: first_row[col] for col in fixed_columns}

        value_to_row = {
            row[factor]: row
            for _, row in group.iterrows()
        }

        for before_val, after_val in transitions:
            if before_val in value_to_row and after_val in value_to_row:
                before_row = value_to_row[before_val]
                after_row = value_to_row[after_val]

                before_metrics = {m: float(before_row[m]) for m in metric_columns}
                after_metrics = {m: float(after_row[m]) for m in metric_columns}
                delta_metrics = {
                    m: after_metrics[m] - before_metrics[m]
                    for m in metric_columns
                }

                results.append({
                    "factor": factor,
                    "tree": tree_value,
                    "transition": (before_val, after_val),
                    "config_key": config_key,
                    "fixed_params": fixed_params,
                    "delta_report": {
                        "before": before_metrics,
                        "after": after_metrics,
                        "delta": delta_metrics
                    }
                })

    return results


def summarize_transition_groups(reports, metric_columns):
    rows = []
    for r in reports:
        row = {
            "factor": r["factor"],
            "tree": r["tree"],
            "transition": f"{r['transition'][0]} -> {r['transition'][1]}",
            "config_key": r["config_key"],
        }
        row.update({f"delta_{m}": r["delta_report"]["delta"][m] for m in metric_columns})
        row.update(r["fixed_params"])
        rows.append(row)

    detailed_df = pd.DataFrame(rows)

    if detailed_df.empty:
        return detailed_df, pd.DataFrame()

    summary_df = (
        detailed_df
        .groupby(["factor", "tree", "transition"], dropna=False)
        [[f"delta_{m}" for m in metric_columns]]
        .agg(["count", "mean", "median", "std"])
        .reset_index()
    )

    # Flatten multi-index columns
    summary_df.columns = [
        "_".join(col).strip("_") if isinstance(col, tuple) else col
        for col in summary_df.columns
    ]

    return detailed_df, summary_df


def analyze_factor_by_tree(df, factor, parameter_columns, metric_columns):
    results = {}
    for tree_value in [False, True]:
        reports = get_transitions_for_factor(
            df=df,
            factor=factor,
            parameter_columns=parameter_columns,
            metric_columns=metric_columns,
            tree_value=tree_value,
        )
        detailed_df, summary_df = summarize_transition_groups(reports, metric_columns)
        results[tree_value] = {
            "reports": reports,
            "detailed": detailed_df,
            "summary": summary_df,
        }
    return results


def analyze_tree_effect(df, parameter_columns, metric_columns):
    """
    Tree must be analyzed globally, not inside a fixed tree stratum.
    """
    reports = get_transitions_for_factor(
        df=df,
        factor="tree",
        parameter_columns=parameter_columns,
        metric_columns=metric_columns,
        tree_value=None,
    )
    detailed_df, summary_df = summarize_transition_groups(reports, metric_columns)
    return {
        "reports": reports,
        "detailed": detailed_df,
        "summary": summary_df,
    }


def analyze_all_factors(df):
    available_parameter_columns = [c for c in FACTOR_SPECS.keys() if c in df.columns]
    work = deduplicate_configs(df, available_parameter_columns)

    out = {}

    # Factors whose impact should be reported separately for tree=False and tree=True
    for factor in available_parameter_columns:
        if factor == "tree":
            continue
        out[factor] = analyze_factor_by_tree(
            df=work,
            factor=factor,
            parameter_columns=available_parameter_columns,
            metric_columns=metric_columns,
        )

    # Separate global analysis for tree itself
    if "tree" in available_parameter_columns:
        out["tree"] = analyze_tree_effect(
            df=work,
            parameter_columns=available_parameter_columns,
            metric_columns=metric_columns,
        )

    return out


In [10]:
# df should already contain one row per experiment result
# with columns like:
# encoder_key, classification_mode, tree, use_edge_type,
# gnn_model_name, path_depth, accuracy, precision_macro, recall_macro, f1_macro

results = analyze_all_factors(df)


In [29]:
def print_summary(summary, factor, tree_value):
    print("-" * 40)
    print(f"{factor} effect when tree={tree_value}:")
    print("-" * 40)
    print(f"Count: {summary['delta_accuracy_count'].values[0]}")
    print(f"Mean: {summary['delta_accuracy_mean'].values[0]:.8f}")
    print(f"Median: {summary['delta_accuracy_median'].values[0]:.8f}")
    print(f"Std: {summary['delta_accuracy_std'].values[0]:.8f}")

    print(f"Mean delta f1_macro: {summary['delta_f1_macro_mean'].values[0]:.8f}")
    print(f"Median delta f1_macro: {summary['delta_f1_macro_median'].values[0]:.8f}")
    print(f"Std delta f1_macro: {summary['delta_f1_macro_std'].values[0]:.8f}")
    
    print()

In [30]:
print_summary(results["classification_mode"][False]["summary"], "Classification mode", False)
print_summary(results["classification_mode"][True]["summary"], "Classification mode", True)

print_summary(results["encoder_key"][False]["summary"], "Text encoder", False)
print_summary(results["encoder_key"][True]["summary"], "Text encoder", True)

print_summary(results["use_edge_type"][False]["summary"], "use_edge_type", False)
print_summary(results["use_edge_type"][True]["summary"], "use_edge_type", True)

print_summary(results["path_depth"][False]["summary"], "path_depth", False)
print_summary(results["path_depth"][True]["summary"], "path_depth", True)

print_summary(results["gnn_model_name"][False]["summary"], "GNN model", False)
print_summary(results["gnn_model_name"][True]["summary"], "GNN model", True)

print_summary(results["tree"]["summary"], "Tree effect", "N/A")

----------------------------------------
Classification mode effect when tree=False:
----------------------------------------
Count: 32
Mean: -0.04507456
Median: -0.04940778
Std: 0.03253695
Mean delta f1_macro: -0.00560114
Median delta f1_macro: -0.02599917
Std delta f1_macro: 0.05745178

----------------------------------------
Classification mode effect when tree=True:
----------------------------------------
Count: 32
Mean: 0.26228585
Median: 0.23282572
Std: 0.14123112
Mean delta f1_macro: 0.32627071
Median delta f1_macro: 0.40706407
Std delta f1_macro: 0.21273846

----------------------------------------
Text encoder effect when tree=False:
----------------------------------------
Count: 32
Mean: -0.00584021
Median: -0.00647208
Std: 0.00490955
Mean delta f1_macro: -0.03377332
Median delta f1_macro: -0.03486344
Std delta f1_macro: 0.00899283

----------------------------------------
Text encoder effect when tree=True:
----------------------------------------
Count: 24
Mean: -0.00226

In [12]:
for factor, factor_result in results.items():
    print(f"\n===== FACTOR: {factor} =====")
    if factor == "tree":
        print(factor_result["summary"])
    else:
        print("\n--- tree = False ---")
        print(factor_result[False]["summary"])
        print("\n--- tree = True ---")
        print(factor_result[True]["summary"])



===== FACTOR: encoder_key =====

--- tree = False ---
        factor   tree           transition  delta_accuracy_count  \
0  encoder_key  False         bow -> tfidf                    32   
1  encoder_key  False         tfidf -> w2v                    32   
2  encoder_key  False  w2v -> bert-encoder                    32   

   delta_accuracy_mean  delta_accuracy_median  delta_accuracy_std  \
0            -0.005840              -0.006472            0.004910   
1            -0.194868              -0.205584            0.025798   
2             0.075074               0.079061            0.024064   

   delta_precision_macro_count  delta_precision_macro_mean  \
0                           32                    0.024321   
1                           32                   -0.225002   
2                           32                    0.103786   

   delta_precision_macro_median  delta_precision_macro_std  \
0                      0.018919                   0.023629   
1                     

In [ ]:
import itertools
import math
import hashlib
from copy import deepcopy

import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display


# -----------------------------------------------------------------------------
# 1. Configuration
# -----------------------------------------------------------------------------

DEFAULTS = {
    "tree": False,                    # Graph
    "encoder_key": "bow",            # BoW
    "use_attributes": False,
    "use_edge_label": False,
    "use_edge_type": False,
    "path_depth": 0,
    "classification_mode": "text",
    "gnn_model_name": "sage",
}

FACTOR_SPECS = {
    "tree": {
        "values": [False, True],
        "label": "Structural Encoding",
    },
    "encoder_key": {
        "values": ["bow", "tfidf", "w2v", "bert-encoder"],
        "label": "Text Encoding",
    },
    "text_config": {
        "values": list(itertools.product([False, True], [False, True], [False, True])),
        "label": "Text Configuration",
    },
    "path_depth": {
        "values": [0, 1, 2, 3],
        "label": "Bag of Paths",
    },
    "classification_mode": {
        "values": ["text", "gnn"],
        "label": "Classification Type",
    },
    "gnn_model_name": {
        "values": ["sage", "gcn", "gat"],
        "label": "GNN Type",
    },
}

STAGE_ORDER = [
    "tree",
    "encoder_key",
    "text_config",
    "path_depth",
    "classification_mode",
    "gnn_model_name",
]

REQUIRED_COLUMNS = [
    "tree",
    "encoder_key",
    "use_attributes",
    "use_edge_label",
    "use_edge_type",
    "path_depth",
    "classification_mode",
    "gnn_model_name",
    "f1_macro",
]

COLOR_SEQ = [
    "#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e",
    "#17becf", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22"
]


# -----------------------------------------------------------------------------
# 2. Helpers
# -----------------------------------------------------------------------------

def prefix_signature(point):
    cfg = point["cfg"]
    return (
        point["stage_idx"],
        bool(cfg["tree"]),
        str(cfg["encoder_key"]),
        bool(cfg["use_attributes"]),
        bool(cfg["use_edge_label"]),
        bool(cfg["use_edge_type"]),
        int(cfg["path_depth"]),
        str(cfg["classification_mode"]),
        str(cfg["gnn_model_name"]),
    )


def build_unique_graph(paths):
    """
    Convert full paths into a DAG-like structure:
    - identical prefix states become one node
    - identical transitions between prefix states become one edge
    """
    node_map = {}
    edge_set = set()

    for path in paths:
        pts = path["points"]
        prev_node_id = None

        for pt in pts:
            node_id = prefix_signature(pt)

            if node_id not in node_map:
                node_map[node_id] = {
                    "node_id": node_id,
                    "stage": pt["stage"],
                    "stage_idx": pt["stage_idx"],
                    "stage_label": pt["stage_label"],
                    "value": pt["value"],
                    "value_label": pt["value_label"],
                    "f1_macro": pt["f1_macro"],
                    "cfg": deepcopy(pt["cfg"]),
                }

            if prev_node_id is not None:
                edge_set.add((prev_node_id, node_id))

            prev_node_id = node_id

    return node_map, edge_set


def node_x_position(node):
    val_id = f"{node['stage']}::{node['value_label']}"
    return node["stage_idx"] + stable_jitter(val_id, scale=0.22)


def node_hover_text(node):
    cfg = node["cfg"]
    return (
        f"<b>{node['stage_label']}</b>: {node['value_label']}<br>"
        f"F1: {node['f1_macro']:.4f}<br>"
        f"tree: {cfg['tree']}<br>"
        f"encoder: {cfg['encoder_key']}<br>"
        f"text_cfg: A{int(cfg['use_attributes'])}-L{int(cfg['use_edge_label'])}-T{int(cfg['use_edge_type'])}<br>"
        f"path_depth: {cfg['path_depth']}<br>"
        f"classification: {cfg['classification_mode']}<br>"
        f"gnn: {cfg['gnn_model_name']}"
    )



def text_config_label(cfg_tuple):
    a, l, t = cfg_tuple
    return f"A{int(a)}-L{int(l)}-T{int(t)}"

def pretty_value(stage, value):
    if stage == "tree":
        return "Tree" if value else "Graph"
    if stage == "encoder_key":
        return {
            "bow": "BoW",
            "tfidf": "TF-IDF",
            "w2v": "W2V",
            "bert-encoder": "BERT",
        }.get(value, str(value))
    if stage == "text_config":
        return text_config_label(value)
    if stage == "path_depth":
        return f"d={value}"
    if stage == "classification_mode":
        return "GNN" if value == "gnn" else "Text"
    if stage == "gnn_model_name":
        return value.upper()
    return str(value)

def normalize_results_df(df):
    work = df.copy()

    for col in REQUIRED_COLUMNS:
        if col not in work.columns:
            if col == "f1_macro":
                raise ValueError("DataFrame must contain 'f1_macro'.")
            work[col] = DEFAULTS[col]

    work["tree"] = work["tree"].astype(bool)
    work["use_attributes"] = work["use_attributes"].astype(bool)
    work["use_edge_label"] = work["use_edge_label"].astype(bool)
    work["use_edge_type"] = work["use_edge_type"].astype(bool)
    work["path_depth"] = work["path_depth"].astype(int)
    work["classification_mode"] = work["classification_mode"].astype(str)
    work["encoder_key"] = work["encoder_key"].astype(str)

    def norm_gnn(row):
        if row["classification_mode"] == "text":
            return "none"
        val = row.get("gnn_model_name", DEFAULTS["gnn_model_name"])
        if pd.isna(val):
            return DEFAULTS["gnn_model_name"]
        return str(val).lower()

    work["gnn_model_name"] = work.apply(norm_gnn, axis=1)

    key_cols = [
        "tree",
        "encoder_key",
        "use_attributes",
        "use_edge_label",
        "use_edge_type",
        "path_depth",
        "classification_mode",
        "gnn_model_name",
    ]
    work = work.drop_duplicates(subset=key_cols, keep="last").copy()
    return work

def config_key(cfg):
    gnn_name = "none" if cfg["classification_mode"] == "text" else cfg["gnn_model_name"]
    return (
        bool(cfg["tree"]),
        str(cfg["encoder_key"]),
        bool(cfg["use_attributes"]),
        bool(cfg["use_edge_label"]),
        bool(cfg["use_edge_type"]),
        int(cfg["path_depth"]),
        str(cfg["classification_mode"]),
        str(gnn_name),
    )

def build_lookup(df):
    lookup = {}
    for _, row in df.iterrows():
        key = (
            bool(row["tree"]),
            str(row["encoder_key"]),
            bool(row["use_attributes"]),
            bool(row["use_edge_label"]),
            bool(row["use_edge_type"]),
            int(row["path_depth"]),
            str(row["classification_mode"]),
            str(row["gnn_model_name"]),
        )
        lookup[key] = row
    return lookup

def apply_stage_value(cfg, stage, value):
    new_cfg = deepcopy(cfg)

    if stage == "tree":
        new_cfg["tree"] = value
    elif stage == "encoder_key":
        new_cfg["encoder_key"] = value
    elif stage == "text_config":
        a, l, t = value
        new_cfg["use_attributes"] = a
        new_cfg["use_edge_label"] = l
        new_cfg["use_edge_type"] = t
    elif stage == "path_depth":
        new_cfg["path_depth"] = value
    elif stage == "classification_mode":
        new_cfg["classification_mode"] = value
        if value == "text":
            new_cfg["gnn_model_name"] = "none"
        else:
            new_cfg["gnn_model_name"] = DEFAULTS["gnn_model_name"]
    elif stage == "gnn_model_name":
        if new_cfg["classification_mode"] == "gnn":
            new_cfg["gnn_model_name"] = value

    if new_cfg["classification_mode"] == "text":
        new_cfg["gnn_model_name"] = "none"

    return new_cfg

def stable_jitter(text, scale=0.16):
    h = hashlib.md5(text.encode()).hexdigest()
    v = int(h[:8], 16) / 0xFFFFFFFF
    return (v - 0.5) * scale

def stage_value_id(stage, value):
    if stage == "text_config":
        return text_config_label(value)
    return str(value)

def generate_paths(df_lookup, selections):
    base_cfg = deepcopy(DEFAULTS)

    active_paths = [{
        "id": "root",
        "cfg": base_cfg,
        "points": [],
    }]

    for stage_idx, stage in enumerate(STAGE_ORDER):
        stage_values = selections[stage]

        new_paths = []
        for path in active_paths:
            if stage == "gnn_model_name" and path["cfg"]["classification_mode"] != "gnn":
                new_paths.append(path)
                continue

            for value in stage_values:
                new_cfg = apply_stage_value(path["cfg"], stage, value)
                row = df_lookup.get(config_key(new_cfg))
                if row is None:
                    continue

                point = {
                    "stage": stage,
                    "stage_idx": stage_idx,
                    "stage_label": FACTOR_SPECS[stage]["label"],
                    "value": value,
                    "value_label": pretty_value(stage, value),
                    "f1_macro": float(row["f1_macro"]),
                    "cfg": deepcopy(new_cfg),
                }

                new_path = {
                    "id": f"{path['id']}|{stage}:{stage_value_id(stage, value)}",
                    "cfg": new_cfg,
                    "points": path["points"] + [point],
                }
                new_paths.append(new_path)

        active_paths = new_paths

    return active_paths

def build_plot(paths, show_labels=True, height=650, dataset = "EA Layer"):
    fig = go.Figure()

    if not paths:
        fig.update_layout(
            title="No valid configurations match the current selections.",
            template="plotly_white",
            height=height
        )
        return fig

    node_map, edge_set = build_unique_graph(paths)

    # Draw each unique edge once
    edge_x = []
    edge_y = []

    for src_id, dst_id in edge_set:
        src = node_map[src_id]
        dst = node_map[dst_id]

        edge_x.extend([node_x_position(src), node_x_position(dst), None])
        edge_y.extend([src["f1_macro"], dst["f1_macro"], None])

    fig.add_trace(go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line=dict(color="rgba(80,80,80,0.45)", width=1.5),
        hoverinfo="skip",
        showlegend=False,
    ))

    # Draw unique nodes once
    nodes_sorted = sorted(node_map.values(), key=lambda n: (n["stage_idx"], n["value_label"]))

    xs = [node_x_position(node) for node in nodes_sorted]
    ys = [node["f1_macro"] for node in nodes_sorted]
    texts = [node["value_label"] if show_labels else "" for node in nodes_sorted]
    hover_texts = [node_hover_text(node) for node in nodes_sorted]

    fig.add_trace(go.Scatter(
        x=xs,
        y=ys,
        mode="markers+text" if show_labels else "markers",
        marker=dict(
            size=9,
            color=ys,
            colorscale="Viridis",
            line=dict(width=0.8, color="white"),
            colorbar=dict(title="F1"),
        ),
        text=texts,
        textposition="top center",
        hovertext=hover_texts,
        hoverinfo="text",
        showlegend=False,
    ))

    for boundary in range(len(STAGE_ORDER) - 1):
        fig.add_vline(
            x=boundary + 0.5,
            line_width=1,
            line_dash="dash",
            line_color="lightgray",
        )

    stage_names = [FACTOR_SPECS[s]["label"] for s in STAGE_ORDER]

    fig.update_layout(
        template="plotly_white",
        height=height,
        title=f"{dataset} Sequential Configuration Transition Plot",
        xaxis=dict(
            tickmode="array",
            tickvals=list(range(len(STAGE_ORDER))),
            ticktext=stage_names,
            title="Configuration Steps",
        ),
        yaxis=dict(
            title="Macro-F1",
            rangemode="tozero",
        ),
        margin=dict(l=40, r=20, t=60, b=80),
    )
    return fig


# -----------------------------------------------------------------------------
# 3. Widgets
# -----------------------------------------------------------------------------

def make_select_multiple(options, default_values, description):
    return widgets.SelectMultiple(
        options=options,
        value=tuple(default_values),
        description=description,
        layout=widgets.Layout(width="260px", height="130px"),
        style={"description_width": "120px"},
    )

tree_widget = make_select_multiple(
    options=[("Graph", False), ("Tree", True)],
    default_values=[False, True],
    description="Structure",
)

encoder_widget = make_select_multiple(
    options=[("BoW", "bow"), ("TF-IDF", "tfidf"), ("W2V", "w2v"), ("BERT", "bert-encoder")],
    default_values=["bow"],
    description="Text Encoder",
)

text_config_options = [(text_config_label(v), v) for v in FACTOR_SPECS["text_config"]["values"]]
text_config_widget = make_select_multiple(
    options=text_config_options,
    default_values=[(False, False, False)],
    description="Text Config",
)

path_depth_widget = make_select_multiple(
    options=[(f"{d}", d) for d in FACTOR_SPECS["path_depth"]["values"]],
    default_values=[0],
    description="Path Length",
)

cls_widget = make_select_multiple(
    options=[("Text", "text"), ("GNN", "gnn")],
    default_values=["text"],
    description="Classification",
)

gnn_widget = make_select_multiple(
    options=[("SAGE", "sage"), ("GCN", "gcn"), ("GAT", "gat")],
    default_values=["sage"],
    description="GNN Type",
)

show_labels_widget = widgets.Checkbox(
    value=True,
    description="Show labels",
)

height_widget = widgets.IntSlider(
    value=650,
    min=400,
    max=1000,
    step=50,
    description="Height",
    style={"description_width": "80px"},
)

output = widgets.Output()


# -----------------------------------------------------------------------------
# 4. Main renderer
# -----------------------------------------------------------------------------

def render_transition_plot(
    tree_values,
    encoder_values,
    text_config_values,
    path_depth_values,
    cls_values,
    gnn_values,
    show_labels,
    height,
    dataset = "EA Layer"
):
    output.clear_output(wait=True)

    selections = {
        "tree": list(tree_values),
        "encoder_key": list(encoder_values),
        "text_config": list(text_config_values),
        "path_depth": list(path_depth_values),
        "classification_mode": list(cls_values),
        "gnn_model_name": list(gnn_values),
    }

    # Fallback to defaults if a widget is emptied
    for stage in selections:
        if not selections[stage]:
            if stage == "text_config":
                selections[stage] = [(False, False, False)]
            else:
                selections[stage] = [DEFAULTS[stage]]

    paths = generate_paths(_DF_LOOKUP, selections)
    fig = build_plot(paths, show_labels=show_labels, height=height, dataset=dataset)

    with output:
        display(fig)


# -----------------------------------------------------------------------------
# 5. Initialize with your dataframe
# -----------------------------------------------------------------------------
# Replace `df` with your actual dataframe variable.

import pandas as pd

df = pd.read_csv("results/analysis_rigorous/evaluation_package/eamodelset_layer/flat_results.csv")
df.drop(columns=['dataset', 'encoder_name', 'target_attr', 'label_attr', 'attributes_attr', 'use_attributes', 'use_edge_label', 'graph_encoder_family', 'structural_encoding_label', 'duplicate_count', 'num_samples', 'num_classes'], inplace=True)
_df_plot = normalize_results_df(df)
_DF_LOOKUP = build_lookup(_df_plot)

controls = widgets.VBox([
    widgets.HBox([tree_widget, encoder_widget, text_config_widget]),
    widgets.HBox([path_depth_widget, cls_widget, gnn_widget]),
    widgets.HBox([show_labels_widget, height_widget]),
])

interactive = widgets.interactive_output(
    render_transition_plot,
    {
        "tree_values": tree_widget,
        "encoder_values": encoder_widget,
        "text_config_values": text_config_widget,
        "path_depth_values": path_depth_widget,
        "cls_values": cls_widget,
        "gnn_values": gnn_widget,
        "show_labels": show_labels_widget,
        "height": height_widget,
        "dataset": "EA Layer"
    }
)

display(controls, output, interactive)

Output()

Output()

In [4]:
df.columns

Index(['encoder_key', 'classification_mode', 'tree', 'path_depth',
       'use_edge_type', 'gnn_model_name', 'accuracy', 'precision_macro',
       'recall_macro', 'f1_macro'],
      dtype='str')

In [7]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


# -----------------------------------------------------------------------------
# 1. Prepare dataframe
# -----------------------------------------------------------------------------

def prepare_correlation_dataframe(df):
    work = df.copy()

    required_base = [
        "encoder_key",
        "classification_mode",
        "tree",
        "use_edge_type",
        "path_depth",
        "gnn_model_name",
        "f1_macro",
    ]
    missing = [c for c in required_base if c not in work.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    optional_binary = ["use_attributes", "use_edge_label"]

    work["tree"] = work["tree"].astype(int)
    work["use_edge_type"] = work["use_edge_type"].astype(int)
    work["path_depth"] = work["path_depth"].astype(int)

    for col in optional_binary:
        if col in work.columns:
            work[col] = work[col].astype(int)

    work["classification_mode"] = work["classification_mode"].astype(str)
    work["encoder_key"] = work["encoder_key"].astype(str)

    def normalize_training_label(row):
        if row["classification_mode"] == "text":
            return "text_cls"
        raw = row.get("gnn_model_name", "sage")
        if pd.isna(raw):
            raw = "sage"
        raw = str(raw).lower()
        return f"gnn_cls_{raw}"

    work["training_label"] = work.apply(normalize_training_label, axis=1)

    dedup_cols = [
        "encoder_key",
        "classification_mode",
        "tree",
        "use_edge_type",
        "path_depth",
        "training_label",
    ]
    for col in optional_binary:
        if col in work.columns:
            dedup_cols.append(col)

    work = work.drop_duplicates(subset=dedup_cols, keep="last").copy()
    return work


# -----------------------------------------------------------------------------
# 2. Encoding matrix per tree stratum
# -----------------------------------------------------------------------------

def build_parameter_matrix(df, tree_value):
    work = prepare_correlation_dataframe(df)
    work = work[work["tree"] == int(tree_value)].copy()

    if work.empty:
        return pd.DataFrame()

    numeric_cols = ["use_edge_type", "path_depth"]
    for col in ["use_attributes", "use_edge_label"]:
        if col in work.columns:
            numeric_cols.append(col)

    numeric_part = work[numeric_cols].copy()

    categorical_part = pd.get_dummies(
        work[["encoder_key", "classification_mode", "training_label"]],
        prefix=["encoder", "cls", "train"],
        dtype=int
    )

    X = pd.concat([numeric_part, categorical_part], axis=1)
    X["f1_macro"] = work["f1_macro"].astype(float)
    return X


# -----------------------------------------------------------------------------
# 3. Labels
# -----------------------------------------------------------------------------

def prettify_corr_label(col):
    mapping = {
        "use_attributes": "Use Attributes",
        "use_edge_label": "Use Edge Label",
        "use_edge_type": "Use Edge Type",
        "path_depth": "Path Length",
        "f1_macro": "F1 Macro",
    }
    if col in mapping:
        return mapping[col]

    if col.startswith("encoder_"):
        v = col.replace("encoder_", "")
        return {
            "bow": "Encoder: BoW",
            "tfidf": "Encoder: TF-IDF",
            "w2v": "Encoder: W2V",
            "bert-encoder": "Encoder: BERT",
        }.get(v, f"Encoder: {v}")

    if col.startswith("cls_"):
        v = col.replace("cls_", "")
        return {
            "text": "Classification: Text",
            "gnn": "Classification: GNN",
        }.get(v, f"Classification: {v}")

    if col.startswith("train_"):
        v = col.replace("train_", "")
        return {
            "text_cls": "Text Cls",
            "gnn_cls_sage": "GNN Cls: SAGE",
            "gnn_cls_gcn": "GNN Cls: GCN",
            "gnn_cls_gat": "GNN Cls: GAT",
        }.get(v, v)

    return col


def tree_title(tree_value):
    return "Tree Encoding" if tree_value else "Graph Encoding"


# -----------------------------------------------------------------------------
# 4. Correlation heatmaps separated by tree
# -----------------------------------------------------------------------------

def plot_parameter_correlation_heatmaps_by_tree(df, method="pearson", focus_on_f1=False, size=1400):
    matrices = {
        False: build_parameter_matrix(df, False),
        True: build_parameter_matrix(df, True),
    }

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[tree_title(False), tree_title(True)],
        horizontal_spacing=0.12,
    )

    showscale_once = True

    for idx, tree_value in enumerate([False, True], start=1):
        X = matrices[tree_value]
        if X.empty:
            continue

        corr = X.corr(method=method)
        corr_plot = corr.loc[["f1_macro"], :] if focus_on_f1 else corr

        x_labels = [prettify_corr_label(c) for c in corr_plot.columns]
        y_labels = [prettify_corr_label(c) for c in corr_plot.index]
        z = corr_plot.values
        text = [[f"{val:.3f}" for val in row] for row in z]

        fig.add_trace(
            go.Heatmap(
                z=z,
                x=x_labels,
                y=y_labels,
                colorscale="RdBu_r",
                zmid=0,
                zmin=-1,
                zmax=1,
                text=text,
                texttemplate="%{text}",
                hovertemplate="X: %{x}<br>Y: %{y}<br>corr: %{z:.4f}<extra></extra>",
                colorbar=dict(title="Correlation") if showscale_once else None,
                showscale=showscale_once,
            ),
            row=1,
            col=idx,
        )
        showscale_once = False

    fig.update_layout(
        template="plotly_white",
        title=f"Parameter Correlation Heatmaps by Structural Encoding ({method.title()})",
        width=size,
        height=420 if focus_on_f1 else 720,
        margin=dict(l=60, r=40, t=80, b=100),
    )
    fig.update_xaxes(tickangle=-45)
    return fig


# -----------------------------------------------------------------------------
# 5. Correlation-to-F1 bars separated by tree
# -----------------------------------------------------------------------------

def plot_f1_correlation_bars_by_tree(df, method="pearson", abs_sort=True, top_n=None):
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[tree_title(False), tree_title(True)],
        horizontal_spacing=0.12,
    )

    for idx, tree_value in enumerate([False, True], start=1):
        X = build_parameter_matrix(df, tree_value)
        if X.empty:
            continue

        corr = X.corr(method=method)["f1_macro"].drop("f1_macro")

        if abs_sort:
            corr = corr.reindex(corr.abs().sort_values(ascending=False).index)
        else:
            corr = corr.sort_values(ascending=False)

        if top_n is not None:
            corr = corr.head(top_n)

        colors = ["#2a9d8f" if v >= 0 else "#e76f51" for v in corr.values]

        fig.add_trace(
            go.Bar(
                x=[prettify_corr_label(c) for c in corr.index],
                y=corr.values,
                marker_color=colors,
                hovertemplate="%{x}<br>corr with F1: %{y:.4f}<extra></extra>",
                showlegend=False,
            ),
            row=1,
            col=idx,
        )

    fig.update_layout(
        template="plotly_white",
        title=f"Correlation of Parameter States with F1 by Structural Encoding ({method.title()})",
        height=550,
        width=1400,
        margin=dict(l=60, r=30, t=80, b=140),
    )
    fig.update_xaxes(tickangle=-45)
    fig.add_hline(y=0, line_width=1, line_color="gray")
    return fig


# -----------------------------------------------------------------------------
# 6. Factor-level effect sizes separated by tree
# -----------------------------------------------------------------------------

def correlation_ratio(categories, values):
    df_tmp = pd.DataFrame({"cat": categories, "val": values}).dropna()
    if df_tmp.empty:
        return np.nan

    grand_mean = df_tmp["val"].mean()
    grouped = df_tmp.groupby("cat")["val"]

    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for _, g in grouped)
    ss_total = ((df_tmp["val"] - grand_mean) ** 2).sum()

    if ss_total == 0:
        return 0.0
    return ss_between / ss_total


def plot_factor_effect_sizes_by_tree(df):
    work = prepare_correlation_dataframe(df)

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=[tree_title(False), tree_title(True)],
        horizontal_spacing=0.12,
    )

    for idx, tree_value in enumerate([False, True], start=1):
        subset = work[work["tree"] == int(tree_value)].copy()
        if subset.empty:
            continue

        factor_defs = {
            "Text Encoder": subset["encoder_key"],
            "Use Edge Type": subset["use_edge_type"].map({0: "False", 1: "True"}),
            "Bag of Paths": subset["path_depth"].astype(str),
            "Classification Type": subset["classification_mode"],
            "Training Setup": subset["training_label"].map({
                "text_cls": "Text Cls",
                "gnn_cls_sage": "GNN Cls: SAGE",
                "gnn_cls_gcn": "GNN Cls: GCN",
                "gnn_cls_gat": "GNN Cls: GAT",
            }).fillna(subset["training_label"]),
        }

        if "use_attributes" in subset.columns:
            factor_defs["Use Attributes"] = subset["use_attributes"].map({0: "False", 1: "True"})
        if "use_edge_label" in subset.columns:
            factor_defs["Use Edge Label"] = subset["use_edge_label"].map({0: "False", 1: "True"})

        scores = []
        for name, cats in factor_defs.items():
            eta2 = correlation_ratio(cats, subset["f1_macro"])
            scores.append({"factor": name, "eta_squared": eta2})

        eff = pd.DataFrame(scores).sort_values("eta_squared", ascending=False)

        fig.add_trace(
            go.Bar(
                x=eff["factor"],
                y=eff["eta_squared"],
                marker_color="#457b9d",
                hovertemplate="%{x}<br>eta²: %{y:.4f}<extra></extra>",
                showlegend=False,
            ),
            row=1,
            col=idx,
        )

    fig.update_layout(
        template="plotly_white",
        title="Factor-Level Association with F1 by Structural Encoding (Eta²)",
        xaxis_title="Factor",
        yaxis_title="Eta²",
        height=520,
        width=1400,
        margin=dict(l=60, r=30, t=80, b=120),
    )
    fig.update_xaxes(tickangle=-30)
    return fig


In [8]:
# Full heatmaps, separate for tree=False and tree=True
fig1 = plot_parameter_correlation_heatmaps_by_tree(df, method="pearson", focus_on_f1=False)
fig1.show()

# Only correlations with F1, separate for tree=False and tree=True
fig2 = plot_parameter_correlation_heatmaps_by_tree(df, method="pearson", focus_on_f1=True)
fig2.show()

# Ranked bars of correlation with F1, separate for tree=False and tree=True
fig3 = plot_f1_correlation_bars_by_tree(df, method="pearson", abs_sort=True)
fig3.show()

# Factor-level effect sizes, separate for tree=False and tree=True
fig4 = plot_factor_effect_sizes_by_tree(df)
fig4.show()


In [9]:
import pandas as pd

df = pd.read_csv("results/analysis_rigorous/evaluation_package/eamodelset_type/flat_results.csv")
df.drop(columns=['dataset', 'encoder_name', 'target_attr', 'label_attr', 'attributes_attr', 'use_attributes', 'use_edge_label', 'graph_encoder_family', 'structural_encoding_label', 'duplicate_count', 'num_samples', 'num_classes'], inplace=True)
_df_plot = normalize_results_df(df)
_DF_LOOKUP = build_lookup(_df_plot)

controls = widgets.VBox([
    widgets.HBox([tree_widget, encoder_widget, text_config_widget]),
    widgets.HBox([path_depth_widget, cls_widget, gnn_widget]),
    widgets.HBox([show_labels_widget, height_widget]),
])

interactive = widgets.interactive_output(
    render_transition_plot,
    {
        "tree_values": tree_widget,
        "encoder_values": encoder_widget,
        "text_config_values": text_config_widget,
        "path_depth_values": path_depth_widget,
        "cls_values": cls_widget,
        "gnn_values": gnn_widget,
        "show_labels": show_labels_widget,
        "height": height_widget,
    }
)

display(controls, output, interactive)

Output()

Output()

In [10]:
# Full heatmaps, separate for tree=False and tree=True
fig1 = plot_parameter_correlation_heatmaps_by_tree(df, method="pearson", focus_on_f1=False)
fig1.show()

# Only correlations with F1, separate for tree=False and tree=True
fig2 = plot_parameter_correlation_heatmaps_by_tree(df, method="pearson", focus_on_f1=True)
fig2.show()

# Ranked bars of correlation with F1, separate for tree=False and tree=True
fig3 = plot_f1_correlation_bars_by_tree(df, method="pearson", abs_sort=True)
fig3.show()

# Factor-level effect sizes, separate for tree=False and tree=True
fig4 = plot_factor_effect_sizes_by_tree(df)
fig4.show()


In [11]:
import pandas as pd

df = pd.read_csv("results/analysis_rigorous/evaluation_package/modelset_abstract/flat_results.csv")
df.drop(columns=['dataset', 'encoder_name', 'target_attr', 'label_attr', 'attributes_attr', 'use_attributes', 'use_edge_label', 'graph_encoder_family', 'structural_encoding_label', 'duplicate_count', 'num_samples', 'num_classes'], inplace=True)
_df_plot = normalize_results_df(df)
_DF_LOOKUP = build_lookup(_df_plot)

controls = widgets.VBox([
    widgets.HBox([tree_widget, encoder_widget, text_config_widget]),
    widgets.HBox([path_depth_widget, cls_widget, gnn_widget]),
    widgets.HBox([show_labels_widget, height_widget]),
])

interactive = widgets.interactive_output(
    render_transition_plot,
    {
        "tree_values": tree_widget,
        "encoder_values": encoder_widget,
        "text_config_values": text_config_widget,
        "path_depth_values": path_depth_widget,
        "cls_values": cls_widget,
        "gnn_values": gnn_widget,
        "show_labels": show_labels_widget,
        "height": height_widget,
    }
)

display(controls, output, interactive)

Output()

Output()

In [12]:
# Full heatmaps, separate for tree=False and tree=True
fig1 = plot_parameter_correlation_heatmaps_by_tree(df, method="pearson", focus_on_f1=False)
fig1.show()

# Only correlations with F1, separate for tree=False and tree=True
fig2 = plot_parameter_correlation_heatmaps_by_tree(df, method="pearson", focus_on_f1=True)
fig2.show()

# Ranked bars of correlation with F1, separate for tree=False and tree=True
fig3 = plot_f1_correlation_bars_by_tree(df, method="pearson", abs_sort=True)
fig3.show()

# Factor-level effect sizes, separate for tree=False and tree=True
fig4 = plot_factor_effect_sizes_by_tree(df)
fig4.show()


In [1]:
import itertools
import hashlib
from copy import deepcopy

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display

In [4]:
# eamodelset_type
df_ea_type = pd.read_csv("results/analysis_rigorous/evaluation_package/eamodelset_type/flat_results.csv")
df_ea_type.drop(
    columns=[
        'dataset', 'encoder_name', 'target_attr', 'label_attr', 'attributes_attr',
        'use_attributes', 'use_edge_label', 'graph_encoder_family',
        'structural_encoding_label', 'duplicate_count', 'num_samples', 'num_classes'
    ],
    inplace=True,
    errors="ignore"
)

df_ea_layer = pd.read_csv("results/analysis_rigorous/evaluation_package/eamodelset_layer/flat_results.csv")
df_ea_layer.drop(
    columns=[
        'dataset', 'encoder_name', 'target_attr', 'label_attr', 'attributes_attr',
        'use_attributes', 'use_edge_label', 'graph_encoder_family',
        'structural_encoding_label', 'duplicate_count', 'num_samples', 'num_classes'
    ],
    inplace=True,
    errors="ignore"
)

# modelset_abstract
df_modelset = pd.read_csv("results/analysis_rigorous/evaluation_package/modelset_abstract/flat_results.csv")
df_modelset.drop(
    columns=[
        'dataset', 'encoder_name', 'target_attr', 'label_attr', 'attributes_attr',
        'graph_encoder_family', 'structural_encoding_label',
        'duplicate_count', 'num_samples', 'num_classes'
    ],
    inplace=True,
    errors="ignore"
)

DATASETS = {
    "EA Type": df_ea_type,
    "EA Layer": df_ea_layer,
    "Modelset Abstract": df_modelset,
}


In [5]:
DEFAULTS = {
    "tree": False,
    "encoder_key": "bow",
    "use_attributes": False,
    "use_edge_label": False,
    "use_edge_type": False,
    "path_depth": 0,
    "classification_mode": "text",
    "gnn_model_name": "sage",
}

COLOR_SEQ = [
    "#1f77b4", "#d62728", "#2ca02c", "#9467bd", "#ff7f0e",
    "#17becf", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22"
]


def normalize_results_df(df):
    work = df.copy()

    required = ["tree", "encoder_key", "use_edge_type", "path_depth", "classification_mode", "gnn_model_name", "f1_macro"]
    missing = [c for c in required if c not in work.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Preserve which text-config dimensions are actually present in the original dataframe
    work.attrs["text_config_dims_present"] = [
        col for col in ["use_attributes", "use_edge_label", "use_edge_type"]
        if col in work.columns
    ]

    for col in ["use_attributes", "use_edge_label"]:
        if col not in work.columns:
            work[col] = False

    work["tree"] = work["tree"].astype(bool)
    work["use_attributes"] = work["use_attributes"].astype(bool)
    work["use_edge_label"] = work["use_edge_label"].astype(bool)
    work["use_edge_type"] = work["use_edge_type"].astype(bool)
    work["path_depth"] = work["path_depth"].astype(int)
    work["classification_mode"] = work["classification_mode"].astype(str)
    work["encoder_key"] = work["encoder_key"].astype(str)

    def norm_gnn(row):
        if row["classification_mode"] == "text":
            return "none"
        val = row.get("gnn_model_name", "sage")
        if pd.isna(val):
            return "sage"
        return str(val).lower()

    work["gnn_model_name"] = work.apply(norm_gnn, axis=1)

    dedup_cols = [
        "tree", "encoder_key", "use_attributes", "use_edge_label", "use_edge_type",
        "path_depth", "classification_mode", "gnn_model_name"
    ]
    work = work.drop_duplicates(subset=dedup_cols, keep="last").copy()

    # preserve attrs after copy
    work.attrs["text_config_dims_present"] = [
        col for col in ["use_attributes", "use_edge_label", "use_edge_type"]
        if col in df.columns
    ]
    return work



def has_use_attributes(df):
    return "use_attributes" in df.columns and df["use_attributes"].nunique() > 1 or "use_attributes" in df.columns

def has_use_edge_label(df):
    return "use_edge_label" in df.columns and df["use_edge_label"].nunique() > 1 or "use_edge_label" in df.columns


def get_text_config_dims(df):
    dims = []
    if "use_attributes" in df.columns:
        dims.append("use_attributes")
    if "use_edge_label" in df.columns:
        dims.append("use_edge_label")
    if "use_edge_type" in df.columns:
        dims.append("use_edge_type")
    return dims


def get_text_config_values(df):
    dims = get_text_config_dims(df)
    return list(itertools.product([False, True], repeat=len(dims))), dims


def text_config_label(cfg_tuple, dims):
    if not dims:
        return "Default"
    parts = []
    for dim, val in zip(dims, cfg_tuple):
        short = {
            "use_attributes": "A",
            "use_edge_label": "L",
            "use_edge_type": "T",
        }[dim]
        parts.append(f"{short}{int(val)}")
    return "-".join(parts)


def apply_text_config(cfg, cfg_tuple, dims):
    new_cfg = deepcopy(cfg)
    for dim, val in zip(dims, cfg_tuple):
        new_cfg[dim] = val
    return new_cfg


def config_key(cfg):
    gnn_name = "none" if cfg["classification_mode"] == "text" else cfg["gnn_model_name"]
    return (
        bool(cfg["tree"]),
        str(cfg["encoder_key"]),
        bool(cfg.get("use_attributes", False)),
        bool(cfg.get("use_edge_label", False)),
        bool(cfg["use_edge_type"]),
        int(cfg["path_depth"]),
        str(cfg["classification_mode"]),
        str(gnn_name),
    )


def build_lookup(df):
    lookup = {}
    for _, row in df.iterrows():
        lookup[(
            bool(row["tree"]),
            str(row["encoder_key"]),
            bool(row.get("use_attributes", False)),
            bool(row.get("use_edge_label", False)),
            bool(row["use_edge_type"]),
            int(row["path_depth"]),
            str(row["classification_mode"]),
            str(row["gnn_model_name"]),
        )] = row
    return lookup


In [6]:
def pretty_stage_value(stage, value, dims=None):
    if stage == "tree":
        return "Tree" if value else "Graph"
    if stage == "encoder_key":
        return {
            "bow": "BoW",
            "tfidf": "TF-IDF",
            "w2v": "W2V",
            "bert-encoder": "BERT",
        }.get(value, str(value))
    if stage == "text_config":
        return text_config_label(value, dims)
    if stage == "path_depth":
        return f"d={value}"
    if stage == "training_setup":
        return {
            "text_cls": "Text Cls",
            "gnn_cls_sage": "GNN Cls: SAGE",
            "gnn_cls_gcn": "GNN Cls: GCN",
            "gnn_cls_gat": "GNN Cls: GAT",
        }.get(value, value)
    return str(value)


def stable_jitter(text, scale=0.18):
    h = hashlib.md5(text.encode()).hexdigest()
    v = int(h[:8], 16) / 0xFFFFFFFF
    return (v - 0.5) * scale


def build_stage_specs(df):
    text_cfg_values, text_cfg_dims = get_text_config_values(df)
    return {
        "tree": {
            "label": "Structural Encoding",
            "values": [False, True],
        },
        "encoder_key": {
            "label": "Text Encoding",
            "values": ["bow", "tfidf", "w2v", "bert-encoder"],
        },
        "text_config": {
            "label": "Text Configuration",
            "values": text_cfg_values,
            "dims": text_cfg_dims,
        },
        "path_depth": {
            "label": "Bag of Paths",
            "values": [0, 1, 2, 3],
        },
        "training_setup": {
            "label": "Training Setup",
            "values": ["text_cls", "gnn_cls_sage", "gnn_cls_gcn", "gnn_cls_gat"],
        },
    }


def apply_stage_value(cfg, stage, value, stage_specs):
    new_cfg = deepcopy(cfg)

    if stage == "tree":
        new_cfg["tree"] = value
    elif stage == "encoder_key":
        new_cfg["encoder_key"] = value
    elif stage == "text_config":
        new_cfg = apply_text_config(new_cfg, value, stage_specs["text_config"]["dims"])
    elif stage == "path_depth":
        new_cfg["path_depth"] = value
    elif stage == "training_setup":
        if value == "text_cls":
            new_cfg["classification_mode"] = "text"
            new_cfg["gnn_model_name"] = "none"
        elif value == "gnn_cls_sage":
            new_cfg["classification_mode"] = "gnn"
            new_cfg["gnn_model_name"] = "sage"
        elif value == "gnn_cls_gcn":
            new_cfg["classification_mode"] = "gnn"
            new_cfg["gnn_model_name"] = "gcn"
        elif value == "gnn_cls_gat":
            new_cfg["classification_mode"] = "gnn"
            new_cfg["gnn_model_name"] = "gat"

    if new_cfg["classification_mode"] == "text":
        new_cfg["gnn_model_name"] = "none"

    return new_cfg


def generate_paths(df_lookup, stage_specs, selections):
    stage_order = list(stage_specs.keys())
    base_cfg = deepcopy(DEFAULTS)

    active_paths = [{"id": "root", "cfg": base_cfg, "points": []}]

    for stage_idx, stage in enumerate(stage_order):
        new_paths = []
        stage_values = selections[stage]

        for path in active_paths:
            for value in stage_values:
                new_cfg = apply_stage_value(path["cfg"], stage, value, stage_specs)
                row = df_lookup.get(config_key(new_cfg))
                if row is None:
                    continue

                point = {
                    "stage": stage,
                    "stage_idx": stage_idx,
                    "stage_label": stage_specs[stage]["label"],
                    "value": value,
                    "value_label": pretty_stage_value(stage, value, stage_specs.get(stage, {}).get("dims")),
                    "f1_macro": float(row["f1_macro"]),
                    "cfg": deepcopy(new_cfg),
                }

                new_paths.append({
                    "id": f"{path['id']}|{stage}:{point['value_label']}",
                    "cfg": new_cfg,
                    "points": path["points"] + [point],
                })

        active_paths = new_paths

    return active_paths


def prefix_signature(point):
    cfg = point["cfg"]
    return (
        point["stage_idx"],
        bool(cfg["tree"]),
        str(cfg["encoder_key"]),
        bool(cfg.get("use_attributes", False)),
        bool(cfg.get("use_edge_label", False)),
        bool(cfg["use_edge_type"]),
        int(cfg["path_depth"]),
        str(cfg["classification_mode"]),
        str(cfg["gnn_model_name"]),
    )


def build_unique_graph(paths):
    node_map = {}
    edge_set = set()

    for path in paths:
        prev_node_id = None
        for pt in path["points"]:
            node_id = prefix_signature(pt)
            if node_id not in node_map:
                node_map[node_id] = {
                    "node_id": node_id,
                    **pt,
                }
            if prev_node_id is not None:
                edge_set.add((prev_node_id, node_id))
            prev_node_id = node_id

    return node_map, edge_set


def node_x_position(node):
    val_id = f"{node['stage']}::{node['value_label']}"
    return node["stage_idx"] + stable_jitter(val_id, scale=0.20)


def node_hover_text(node, dims):
    cfg = node["cfg"]
    text_cfg = text_config_label(tuple(cfg[d] for d in dims), dims) if dims else "Default"
    train_label = "Text Cls" if cfg["classification_mode"] == "text" else f"GNN Cls: {cfg['gnn_model_name'].upper()}"

    return (
        f"<b>{node['stage_label']}</b>: {node['value_label']}<br>"
        f"F1: {node['f1_macro']:.4f}<br>"
        f"Structure: {'Tree' if cfg['tree'] else 'Graph'}<br>"
        f"Encoder: {cfg['encoder_key']}<br>"
        f"Text Config: {text_cfg}<br>"
        f"Path Length: {cfg['path_depth']}<br>"
        f"Training: {train_label}"
    )


def build_sequential_plot(df, selections, show_labels=True, height=650):
    df_norm = normalize_results_df(df)
    stage_specs = build_stage_specs(df_norm)
    df_lookup = build_lookup(df_norm)
    paths = generate_paths(df_lookup, stage_specs, selections)

    fig = go.Figure()

    if not paths:
        fig.update_layout(
            title="No valid configurations match the current selections.",
            template="plotly_white",
            height=height,
        )
        return fig

    node_map, edge_set = build_unique_graph(paths)

    edge_x = []
    edge_y = []

    for src_id, dst_id in edge_set:
        src = node_map[src_id]
        dst = node_map[dst_id]
        edge_x.extend([node_x_position(src), node_x_position(dst), None])
        edge_y.extend([src["f1_macro"], dst["f1_macro"], None])

    fig.add_trace(go.Scatter(
        x=edge_x,
        y=edge_y,
        mode="lines",
        line=dict(color="rgba(70,70,70,0.45)", width=1.6),
        hoverinfo="skip",
        showlegend=False,
    ))

    dims = stage_specs["text_config"]["dims"]
    nodes_sorted = sorted(node_map.values(), key=lambda n: (n["stage_idx"], n["value_label"]))

    xs = [node_x_position(n) for n in nodes_sorted]
    ys = [n["f1_macro"] for n in nodes_sorted]
    texts = [n["value_label"] if show_labels else "" for n in nodes_sorted]
    hover_texts = [node_hover_text(n, dims) for n in nodes_sorted]

    fig.add_trace(go.Scatter(
        x=xs,
        y=ys,
        mode="markers+text" if show_labels else "markers",
        marker=dict(
            size=9,
            color=ys,
            colorscale="Viridis",
            line=dict(width=0.8, color="white"),
            colorbar=dict(title="F1"),
        ),
        text=texts,
        textposition="top center",
        hovertext=hover_texts,
        hoverinfo="text",
        showlegend=False,
    ))

    stage_order = list(stage_specs.keys())
    for boundary in range(len(stage_order) - 1):
        fig.add_vline(x=boundary + 0.5, line_width=1, line_dash="dash", line_color="lightgray")

    fig.update_layout(
        template="plotly_white",
        title="Sequential Configuration Transition Plot",
        height=height,
        xaxis=dict(
            tickmode="array",
            tickvals=list(range(len(stage_order))),
            ticktext=[stage_specs[s]["label"] for s in stage_order],
            title="Configuration Steps",
        ),
        yaxis=dict(title="Macro-F1", rangemode="tozero"),
        margin=dict(l=40, r=20, t=60, b=80),
    )
    return fig


In [7]:
def make_sequential_plot_widget(df):
    df_norm = normalize_results_df(df)
    stage_specs = build_stage_specs(df_norm)

    tree_widget = widgets.SelectMultiple(
        options=[("Graph", False), ("Tree", True)],
        value=(False, True),
        description="Structure",
        layout=widgets.Layout(width="220px", height="110px"),
        style={"description_width": "90px"},
    )

    encoder_widget = widgets.SelectMultiple(
        options=[("BoW", "bow"), ("TF-IDF", "tfidf"), ("W2V", "w2v"), ("BERT", "bert-encoder")],
        value=("bow",),
        description="Encoder",
        layout=widgets.Layout(width="220px", height="130px"),
        style={"description_width": "90px"},
    )

    text_cfg_widget = widgets.SelectMultiple(
        options=[(text_config_label(v, stage_specs["text_config"]["dims"]), v) for v in stage_specs["text_config"]["values"]],
        value=(stage_specs["text_config"]["values"][0],),
        description="Text Config",
        layout=widgets.Layout(width="260px", height="160px"),
        style={"description_width": "90px"},
    )

    path_widget = widgets.SelectMultiple(
        options=[(str(v), v) for v in stage_specs["path_depth"]["values"]],
        value=(0,),
        description="Path Len",
        layout=widgets.Layout(width="180px", height="120px"),
        style={"description_width": "90px"},
    )

    training_widget = widgets.SelectMultiple(
        options=[
            ("Text Cls", "text_cls"),
            ("GNN Cls: SAGE", "gnn_cls_sage"),
            ("GNN Cls: GCN", "gnn_cls_gcn"),
            ("GNN Cls: GAT", "gnn_cls_gat"),
        ],
        value=("text_cls",),
        description="Training",
        layout=widgets.Layout(width="240px", height="140px"),
        style={"description_width": "90px"},
    )

    show_labels_widget = widgets.Checkbox(value=True, description="Show labels")
    height_widget = widgets.IntSlider(value=650, min=400, max=1000, step=50, description="Height")

    output = widgets.Output()

    def render(tree_vals, enc_vals, txt_vals, path_vals, train_vals, show_labels, height):
        output.clear_output(wait=True)

        selections = {
            "tree": list(tree_vals) or [False],
            "encoder_key": list(enc_vals) or ["bow"],
            "text_config": list(txt_vals) or [stage_specs["text_config"]["values"][0]],
            "path_depth": list(path_vals) or [0],
            "training_setup": list(train_vals) or ["text_cls"],
        }

        fig = build_sequential_plot(df_norm, selections, show_labels=show_labels, height=height)
        with output:
            display(fig)

    interactive = widgets.interactive_output(
        render,
        {
            "tree_vals": tree_widget,
            "enc_vals": encoder_widget,
            "txt_vals": text_cfg_widget,
            "path_vals": path_widget,
            "train_vals": training_widget,
            "show_labels": show_labels_widget,
            "height": height_widget,
        }
    )

    controls = widgets.VBox([
        widgets.HBox([tree_widget, encoder_widget, text_cfg_widget]),
        widgets.HBox([path_widget, training_widget]),
        widgets.HBox([show_labels_widget, height_widget]),
    ])

    display(controls, output, interactive)


In [8]:
def prepare_ordered_factor_dataframe(df):
    work = normalize_results_df(df)

    dims = get_text_config_dims(work)
    work["text_config_group"] = [
        text_config_label(tuple(row[d] for d in dims), dims)
        for _, row in work.iterrows()
    ]

    def training_setup_label(row):
        if row["classification_mode"] == "text":
            return "Text Cls"
        return f"GNN Cls: {row['gnn_model_name'].upper()}"

    work["training_setup"] = work.apply(training_setup_label, axis=1)
    return work, dims


def correlation_ratio(categories, values):
    tmp = pd.DataFrame({"cat": categories, "val": values}).dropna()
    if tmp.empty:
        return np.nan

    grand_mean = tmp["val"].mean()
    grouped = tmp.groupby("cat")["val"]

    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for _, g in grouped)
    ss_total = ((tmp["val"] - grand_mean) ** 2).sum()
    if ss_total == 0:
        return 0.0
    return ss_between / ss_total


def ordered_factor_scores(df, tree_value):
    work, dims = prepare_ordered_factor_dataframe(df)
    subset = work[work["tree"] == tree_value].copy()

    factor_defs = {
        "Text Encoder": subset["encoder_key"],
        "Text Configuration": subset["text_config_group"],
        "Bag of Paths": subset["path_depth"].astype(str),
        "Training Setup": subset["training_setup"],
    }

    scores = []
    for factor_name, series in factor_defs.items():
        eta2 = correlation_ratio(series, subset["f1_macro"])
        scores.append({"factor": factor_name, "eta_squared": eta2})

    return pd.DataFrame(scores)


def plot_ordered_factor_effect_sizes_by_tree(df):
    left = ordered_factor_scores(df, False)
    right = ordered_factor_scores(df, True)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Graph Encoding", "Tree Encoding"],
        horizontal_spacing=0.12
    )

    for idx, eff in enumerate([left, right], start=1):
        fig.add_trace(
            go.Bar(
                x=eff["factor"],
                y=eff["eta_squared"],
                marker_color="#457b9d",
                hovertemplate="%{x}<br>eta²: %{y:.4f}<extra></extra>",
                showlegend=False,
            ),
            row=1, col=idx
        )

    fig.update_layout(
        template="plotly_white",
        title="Ordered Factor Association with F1 by Structural Encoding",
        height=500,
        width=1200,
        margin=dict(l=60, r=30, t=70, b=90),
    )
    return fig


In [9]:
def build_ordered_state_matrix(df, tree_value):
    work, dims = prepare_ordered_factor_dataframe(df)
    work = work[work["tree"] == tree_value].copy()
    if work.empty:
        return pd.DataFrame()

    parts = []

    enc = pd.get_dummies(work["encoder_key"], prefix="Encoder", dtype=int)
    parts.append(enc)

    txt = pd.get_dummies(work["text_config_group"], prefix="TextCfg", dtype=int)
    parts.append(txt)

    path = pd.get_dummies(work["path_depth"].astype(str), prefix="Path", dtype=int)
    parts.append(path)

    train = pd.get_dummies(work["training_setup"], prefix="Train", dtype=int)
    parts.append(train)

    X = pd.concat(parts, axis=1)
    X["f1_macro"] = work["f1_macro"].astype(float)
    return X


def prettify_ordered_state_label(col):
    
    if col.startswith("Encoder_"):
        return col.replace("Encoder_", "Encoder: ")
    if col.startswith("TextCfg_"):
        return col.replace("TextCfg_", "Text Config: ")
    if col.startswith("Path_"):
        return col.replace("Path_", "Path: ")
    if col.startswith("Train_"):
        return col.replace("Train_", "")
    if col == "f1_macro":
        return "F1 Macro"
    return col


def plot_ordered_state_correlations_by_tree(df, method="pearson"):
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=["Graph Encoding", "Tree Encoding"],
        horizontal_spacing=0.12
    )

    show_scale = True
    for idx, tree_value in enumerate([False, True], start=1):
        X = build_ordered_state_matrix(df, tree_value)
        if X.empty:
            continue

        corr = X.corr(method=method).loc[["f1_macro"], :]
        x_labels = [prettify_ordered_state_label(c) for c in corr.columns]
        y_labels = ["F1 Macro"]
        z = corr.values
        # text = [[f"{v:.3f}" for v in row] for row in z]

        fig.add_trace(
            go.Heatmap(
                z=z,
                x=x_labels,
                y=y_labels,
                colorscale="RdBu_r",
                zmid=0,
                zmin=-1,
                zmax=1,
                # text=text,
                texttemplate="%{text}",
                hovertemplate="State: %{x}<br>corr with F1: %{z:.4f}<extra></extra>",
                showscale=show_scale,
                colorbar=dict(title="Correlation") if show_scale else None,
            ),
            row=1, col=idx
        )
        show_scale = False

    fig.update_layout(
        template="plotly_white",
        title=f"Ordered State Correlations with F1 by Structural Encoding ({method.title()})",
        height=380,
        width=1400,
        margin=dict(l=60, r=30, t=75, b=120),
    )
    fig.update_xaxes(tickangle=-45)
    return fig


In [8]:
def plot_all_for_dataset(name, df):
    print(f"\n===== {name} =====")

    # Sequential interactive config plot
    make_sequential_plot_widget(df)

    # Ordered factor importance
    fig1 = plot_ordered_factor_effect_sizes_by_tree(df)
    fig1.show()

    # Ordered state-vs-F1 correlations
    fig2 = plot_ordered_state_correlations_by_tree(df, method="pearson")
    fig2.show()


for name, df_ in DATASETS.items():
    plot_all_for_dataset(name, df_)



===== EA Type =====


Output()

Output()


===== EA Layer =====


Output()

Output()


===== Modelset Abstract =====


Output()

Output()

In [9]:
for d, df_ in DATASETS.items():
    ### remove duplicates based on config columns, keeping the highest f1
    config_cols = ["tree", "encoder_key", "use_edge_type", "path_depth", "classification_mode", "gnn_model_name"]
    if "use_edge_label" in df_.columns:
        config_cols.append("use_edge_label")
    if "use_attributes" in df_.columns:
        config_cols.append("use_attributes")
    df_ = df_.sort_values("f1_macro", ascending=False).drop_duplicates(subset=config_cols, keep="first")
    df_.to_excel(f"results/analysis_rigorous/evaluation_package/{d}_deduped.xlsx", index=False)
    

In [10]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def _normalize_plot_df(df):
    work = df.copy()

    required = [
        "encoder_key", "classification_mode", "tree",
        "use_edge_type", "path_depth", "gnn_model_name", "f1_macro"
    ]
    missing = [c for c in required if c not in work.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    if "use_attributes" not in work.columns:
        work["use_attributes"] = False
    if "use_edge_label" not in work.columns:
        work["use_edge_label"] = False

    work["tree"] = work["tree"].astype(bool)
    work["use_attributes"] = work["use_attributes"].astype(bool)
    work["use_edge_label"] = work["use_edge_label"].astype(bool)
    work["use_edge_type"] = work["use_edge_type"].astype(bool)
    work["path_depth"] = work["path_depth"].astype(int)
    work["classification_mode"] = work["classification_mode"].astype(str)
    work["encoder_key"] = work["encoder_key"].astype(str)

    def normalize_gnn(row):
        if row["classification_mode"] == "text":
            return "none"
        val = row.get("gnn_model_name", "sage")
        if pd.isna(val):
            return "sage"
        val = str(val).lower()
        if val == "unknown":
            return "sage"
        return val

    work["gnn_model_name"] = work.apply(normalize_gnn, axis=1)
    return work


def _pretty_encoder_label(value):
    mapping = {
        "bow": "Encoder: BoW",
        "tfidf": "Encoder: TFIDF",
        "w2v": "Encoder: Word2vec",
        "bert-encoder": "Encoder: BERT",
        "bert": "Encoder: BERT",
    }
    return mapping.get(value, f"Encoder: {value}")


def _pretty_gnn_label(value):
    mapping = {
        "sage": "GNN: SAGE",
        "gcn": "GNN: GCN",
        "gat": "GNN: GAT",
        "unknown": "GNN: SAGE",
    }
    return mapping.get(value, f"GNN: {str(value).upper()}")


def _state_matrix(df, tree_value):
    work = _normalize_plot_df(df)
    work = work[work["tree"] == tree_value].copy()
    if work.empty:
        return pd.DataFrame()

    parts = []

    # Encoder states
    enc_labels = work["encoder_key"].map(_pretty_encoder_label)
    enc = pd.get_dummies(enc_labels, dtype=int)
    parts.append(enc)

    # Only keep Classification: Text
    cls_text = (work["classification_mode"] == "text").astype(int)
    cls_text.name = "Classification: Text"
    parts.append(cls_text.to_frame())

    # GNN states only for actual GNN runs
    gnn_labels = work.apply(
        lambda r: _pretty_gnn_label(r["gnn_model_name"]) if r["classification_mode"] == "gnn" else None,
        axis=1
    )
    gnn = pd.get_dummies(gnn_labels, dtype=int)
    if not gnn.empty:
        parts.append(gnn)

    # Numeric states
    path_part = work[["path_depth"]].rename(columns={"path_depth": "Path Length"})
    parts.append(path_part)

    edge_part = work[["use_edge_type"]].astype(int).rename(columns={"use_edge_type": "Use Edge Type"})
    parts.append(edge_part)

    X = pd.concat(parts, axis=1)
    X["f1_macro"] = work["f1_macro"].astype(float)

    # Drop constant columns
    constant_cols = [c for c in X.columns if c != "f1_macro" and X[c].nunique() <= 1]
    X = X.drop(columns=constant_cols, errors="ignore")

    return X


def plot_parameter_state_correlations_by_tree(df, dataset_name, method="pearson"):
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["Graph Encoding", "Tree Encoding"],
        horizontal_spacing=0.10,
    )

    for idx, tree_value in enumerate([False, True], start=1):
        X = _state_matrix(df, tree_value)
        if X.empty:
            continue

        corr = X.corr(method=method)["f1_macro"].drop("f1_macro")
        corr = corr.reindex(corr.abs().sort_values(ascending=False).index)

        colors = ["#2a9d8f" if v >= 0 else "#e76f51" for v in corr.values]

        fig.add_trace(
            go.Bar(
                x=list(corr.index),
                y=list(corr.values),
                marker_color=colors,
                hovertemplate="%{x}<br>corr with F1: %{y:.4f}<extra></extra>",
                showlegend=False,
            ),
            row=1,
            col=idx,
        )

    fig.update_layout(
        template="plotly_white",
        title=f"{dataset_name}: Correlation of Parameter States with F1 by Structural Encoding ({method.title()})",
        height=650,
        width=1450,
        margin=dict(l=60, r=30, t=95, b=160),
    )
    fig.update_xaxes(tickangle=-45)
    fig.add_hline(y=0, line_width=1, line_color="gray")
    return fig


In [11]:
def _correlation_ratio(categories, values):
    tmp = pd.DataFrame({"cat": categories, "val": values}).dropna()
    if tmp.empty:
        return np.nan

    grand_mean = tmp["val"].mean()
    grouped = tmp.groupby("cat")["val"]

    ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for _, g in grouped)
    ss_total = ((tmp["val"] - grand_mean) ** 2).sum()

    if ss_total == 0:
        return 0.0
    return ss_between / ss_total


def _factor_scores(df, tree_value):
    work = _normalize_plot_df(df)
    work = work[work["tree"] == tree_value].copy()
    if work.empty:
        return pd.DataFrame(columns=["factor", "eta_squared"])

    factors = {
        "Text Encoder": work["encoder_key"],
        "Training Setup": work.apply(
            lambda r: "Text Cls" if r["classification_mode"] == "text" else f"GNN Cls: {r['gnn_model_name'].upper()}",
            axis=1
        ),
        "Classification Type": work["classification_mode"].map({"text": "Text", "gnn": "GNN"}),
        "Bag of Paths": work["path_depth"].astype(str),
        "Use Edge Type": work["use_edge_type"].map({False: "False", True: "True"}),
    }

    if "use_attributes" in df.columns:
        factors["Use Attributes"] = work["use_attributes"].map({False: "False", True: "True"})
    if "use_edge_label" in df.columns:
        factors["Use Edge Label"] = work["use_edge_label"].map({False: "False", True: "True"})

    rows = []
    for factor_name, cats in factors.items():
        rows.append({
            "factor": factor_name,
            "eta_squared": _correlation_ratio(cats, work["f1_macro"])
        })

    out = pd.DataFrame(rows).sort_values("eta_squared", ascending=False)
    return out


def plot_factor_level_association_by_tree(df, dataset_name):
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["Graph Encoding", "Tree Encoding"],
        horizontal_spacing=0.10,
    )

    for idx, tree_value in enumerate([False, True], start=1):
        eff = _factor_scores(df, tree_value)

        fig.add_trace(
            go.Bar(
                x=eff["factor"],
                y=eff["eta_squared"],
                marker_color="#4f81a3",
                hovertemplate="%{x}<br>Eta²: %{y:.4f}<extra></extra>",
                showlegend=False,
            ),
            row=1,
            col=idx,
        )

    fig.update_layout(
        template="plotly_white",
        title=f"{dataset_name}: Factor-Level Association with F1 by Structural Encoding (Eta²)",
        height=650,
        width=1450,
        margin=dict(l=60, r=30, t=95, b=140),
    )
    fig.update_xaxes(tickangle=-30)
    fig.update_yaxes(title_text="Eta²")
    return fig


In [12]:
# EA Layer
fig1 = plot_parameter_state_correlations_by_tree(df_ea_layer, "EA Layer", method="pearson")
fig1.show()

fig2 = plot_factor_level_association_by_tree(df_ea_layer, "EA Layer")
fig2.show()

# EA Type
fig3 = plot_parameter_state_correlations_by_tree(df_ea_type, "EA Type", method="pearson")
fig3.show()

fig4 = plot_factor_level_association_by_tree(df_ea_type, "EA Type")
fig4.show()

# Modelset Abstract
fig5 = plot_parameter_state_correlations_by_tree(df_modelset, "Modelset Abstract", method="pearson")
fig5.show()

fig6 = plot_factor_level_association_by_tree(df_modelset, "Modelset Abstract")
fig6.show()


In [13]:
ALL_DATASETS = {
    "EA Layer": df_ea_layer,
    "EA Type": df_ea_type,
    "Modelset Abstract": df_modelset,
}

for name, df_ in ALL_DATASETS.items():
    plot_parameter_state_correlations_by_tree(df_, name, method="pearson").show()
    plot_factor_level_association_by_tree(df_, name).show()


In [14]:
import os

out_dir = "results/analysis_rigorous/plots"
os.makedirs(out_dir, exist_ok=True)

for name, df_ in ALL_DATASETS.items():
    safe = name.lower().replace(" ", "_")
    fig_a = plot_parameter_state_correlations_by_tree(df_, name, method="pearson")
    fig_b = plot_factor_level_association_by_tree(df_, name)

    fig_a.write_html(f"{out_dir}/{safe}_state_correlations.html")
    fig_b.write_html(f"{out_dir}/{safe}_factor_association.html")


In [15]:
PLOT_FONT_FAMILY = "Arial"
PLOT_TITLE_SIZE = 28
PLOT_SUBTITLE_SIZE = 22
PLOT_AXIS_TITLE_SIZE = 22
PLOT_TICK_SIZE = 18
PLOT_LEGEND_SIZE = 18
PLOT_ANNOT_SIZE = 16

PUB_WIDTH = 1700
PUB_HEIGHT_BAR = 780
PUB_HEIGHT_FACTOR = 780


def _apply_publication_layout(fig, title, width, height, bottom_margin=180):
    fig.update_layout(
        template="plotly_white",
        title=dict(
            text=title,
            x=0.5,
            xanchor="center",
            font=dict(size=PLOT_TITLE_SIZE, family=PLOT_FONT_FAMILY),
        ),
        width=width,
        height=height,
        font=dict(
            family=PLOT_FONT_FAMILY,
            size=PLOT_TICK_SIZE,
            color="#243b63",
        ),
        margin=dict(l=90, r=40, t=110, b=bottom_margin),
        paper_bgcolor="white",
        plot_bgcolor="white",
    )

    fig.update_xaxes(
        showline=True,
        linewidth=1.2,
        linecolor="#6e7b8b",
        mirror=False,
        ticks="outside",
        tickfont=dict(size=PLOT_TICK_SIZE, family=PLOT_FONT_FAMILY),
        title_font=dict(size=PLOT_AXIS_TITLE_SIZE, family=PLOT_FONT_FAMILY),
        gridcolor="#dfe7f3",
        zeroline=False,
    )

    fig.update_yaxes(
        showline=True,
        linewidth=1.2,
        linecolor="#6e7b8b",
        mirror=False,
        ticks="outside",
        tickfont=dict(size=PLOT_TICK_SIZE, family=PLOT_FONT_FAMILY),
        title_font=dict(size=PLOT_AXIS_TITLE_SIZE, family=PLOT_FONT_FAMILY),
        gridcolor="#dfe7f3",
        zeroline=False,
    )

    for ann in fig.layout.annotations:
        ann.font = dict(size=PLOT_SUBTITLE_SIZE, family=PLOT_FONT_FAMILY, color="#243b63")

    return fig


In [16]:
def plot_parameter_state_correlations_by_tree_pub(df, dataset_name, method="pearson"):
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["Graph Encoding", "Tree Encoding"],
        horizontal_spacing=0.10,
    )

    for idx, tree_value in enumerate([False, True], start=1):
        X = _state_matrix(df, tree_value)
        if X.empty:
            continue

        corr = X.corr(method=method)["f1_macro"].drop("f1_macro")
        corr = corr.reindex(corr.abs().sort_values(ascending=False).index)

        colors = ["#2a9d8f" if v >= 0 else "#e76f51" for v in corr.values]

        fig.add_trace(
            go.Bar(
                x=list(corr.index),
                y=list(corr.values),
                marker=dict(
                    color=colors,
                    line=dict(color="rgba(0,0,0,0.18)", width=0.6),
                ),
                hovertemplate="%{x}<br>corr with F1: %{y:.4f}<extra></extra>",
                showlegend=False,
            ),
            row=1,
            col=idx,
        )

    fig.add_hline(y=0, line_width=1.5, line_color="#6e7b8b")

    fig.update_xaxes(tickangle=-45)
    fig.update_yaxes(title_text="Pearson Correlation")

    _apply_publication_layout(
        fig,
        title=f"{dataset_name}: Correlation of Parameter States with F1 by Structural Encoding ({method.title()})",
        width=PUB_WIDTH,
        height=PUB_HEIGHT_BAR,
        bottom_margin=210,
    )
    return fig


In [17]:
def plot_factor_level_association_by_tree_pub(df, dataset_name):
    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["Graph Encoding", "Tree Encoding"],
        horizontal_spacing=0.10,
    )

    for idx, tree_value in enumerate([False, True], start=1):
        eff = _factor_scores(df, tree_value)

        fig.add_trace(
            go.Bar(
                x=eff["factor"],
                y=eff["eta_squared"],
                marker=dict(
                    color="#4f81a3",
                    line=dict(color="rgba(0,0,0,0.18)", width=0.6),
                ),
                hovertemplate="%{x}<br>Eta²: %{y:.4f}<extra></extra>",
                showlegend=False,
            ),
            row=1,
            col=idx,
        )

    fig.update_xaxes(tickangle=-30)
    fig.update_yaxes(title_text="Eta²")

    _apply_publication_layout(
        fig,
        title=f"{dataset_name}: Factor-Level Association with F1 by Structural Encoding (Eta²)",
        width=PUB_WIDTH,
        height=PUB_HEIGHT_FACTOR,
        bottom_margin=170,
    )
    return fig


In [19]:
ALL_DATASETS = {
    "EA Layer": df_ea_layer,
    "EA Type": df_ea_type,
    "Modelset Abstract": df_modelset,
}

for name, df_ in ALL_DATASETS.items():
    safe = name.lower().replace(" ", "_")

    fig1 = plot_parameter_state_correlations_by_tree_pub(df_, name)
    fig2 = plot_factor_level_association_by_tree_pub(df_, name)

    fig1.write_image(
        f"results/analysis_rigorous/plots/{safe}_state_correlations_pub.png",
        width=PUB_WIDTH,
        height=PUB_HEIGHT_BAR,
        scale=2
    )
    fig2.write_image(
        f"results/analysis_rigorous/plots/{safe}_factor_association_pub.png",
        width=PUB_WIDTH,
        height=PUB_HEIGHT_FACTOR,
        scale=2
    )
